In [8]:
import pandas as pd
import sqlite3

In [9]:
df = pd.read_csv("../data/processed/loan_cleaned.csv")

print("Shape:", df.shape)
df.head()

Shape: (1303638, 92)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,annual_inc,dti,delinq_2yrs,...,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY,application_type_Joint App,loan_to_income,installment_to_income,open_to_total_accounts,credit_issue_count
0,30000,36,22.35,1151.16,4,20,5.0,100000.0,30.46,0.0,...,0,0,0,0,0,1,0.299997,0.011511,0.550000,1.0
1,40000,60,16.14,975.71,3,14,0.0,45000.0,50.53,0.0,...,0,0,0,0,0,1,0.888869,0.021682,0.473684,0.0
2,20000,36,7.56,622.68,1,3,10.0,100000.0,18.92,0.0,...,0,1,0,0,0,1,0.199998,0.006227,0.450000,0.0
3,4500,36,11.31,147.99,2,8,10.0,38500.0,4.64,0.0,...,0,0,0,0,0,0,0.116880,0.003844,0.461538,0.0
4,8425,36,27.27,345.18,5,25,3.0,450000.0,12.37,0.0,...,0,0,0,0,0,1,0.018722,0.000767,0.552632,0.0


In [10]:
conn = sqlite3.connect("../data/processed/credit_risk.db")

print("Database connected successfully!")

Database connected successfully!


In [11]:
df.to_sql(
    "loans",
    conn,
    if_exists="replace",
    index=False
)

print("Loans table created successfully!")

Loans table created successfully!


In [12]:
query = """
SELECT COUNT(*) AS total_records
FROM loans;
"""

pd.read_sql_query(query, conn)

,total_records
0,1303638


In [13]:
query = """
PRAGMA table_info(loans);
"""

pd.read_sql_query(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,loan_amnt,INTEGER,0,None,0
1,1,term,INTEGER,0,None,0
2,2,int_rate,REAL,0,None,0
3,3,installment,REAL,0,None,0
4,4,grade,INTEGER,0,None,0
...,...,...,...,...,...,...
87,87,application_type_Joint App,INTEGER,0,None,0
88,88,loan_to_income,REAL,0,None,0
89,89,installment_to_income,REAL,0,None,0
90,90,open_to_total_accounts,REAL,0,None,0


Query 1 — Loan status distribution

In [17]:
query = """
SELECT 
    "default",
    COUNT(*) AS total_loans
FROM loans
GROUP BY "default"
ORDER BY "default";
"""

pd.read_sql_query(query, conn)

,default,total_loans
0,0,1041952
1,1,261686


Query 2 — Overall default rate

In [20]:
query = """
SELECT 
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate
FROM loans;
"""

pd.read_sql_query(query, conn)

,total_loans,defaulted_loans,default_rate
0,1303638,261686,20.07


Query 3 — Default Rate by Grade

In [19]:
query = """
SELECT 
    grade,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate
FROM loans
GROUP BY grade
ORDER BY grade;
"""

pd.read_sql_query(query, conn)

,grade,total_loans,defaulted_loans,default_rate
0,1,226245,13768,6.09
1,2,380158,51083,13.44
2,3,369937,83271,22.51
3,4,195288,59449,30.44
4,5,91574,35368,38.62
5,6,31485,14265,45.31
6,7,8951,4482,50.07


### Business Insight

Default rate is analyzed across loan grades to identify whether lower credit grades are associated with higher default risk.

In [22]:
df_sql = pd.read_csv("../data/processed/loan_selected.csv")

print(df_sql.shape)

(1303638, 23)


In [23]:
df_sql.to_sql(
    "loans",
    conn,
    if_exists="replace",
    index=False
)

print("SQL table updated successfully!")

SQL table updated successfully!


In [24]:
pd.read_sql_query("""
PRAGMA table_info(loans);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,loan_amnt,INTEGER,0,None,0
1,1,term,TEXT,0,None,0
2,2,int_rate,REAL,0,None,0
3,3,installment,REAL,0,None,0
4,4,grade,TEXT,0,None,0
5,5,sub_grade,TEXT,0,None,0
6,6,emp_length,TEXT,0,None,0
7,7,home_ownership,TEXT,0,None,0
8,8,annual_inc,REAL,0,None,0
9,9,verification_status,TEXT,0,None,0


Query 4 — Default Rate by Purpose

In [26]:
query = """
SELECT 
    purpose,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate
FROM loans
GROUP BY purpose
HAVING COUNT(*) >= 1000
ORDER BY default_rate DESC;
"""

pd.read_sql_query(query, conn)

,purpose,total_loans,defaulted_loans,default_rate
0,small_business,15010,4465,29.75
1,moving,9173,2151,23.45
2,medical,15024,3292,21.91
3,house,6967,1513,21.72
4,debt_consolidation,757610,161058,21.26
5,other,74937,15867,21.17
6,vacation,8732,1680,19.24
7,major_purchase,28328,5304,18.72
8,home_improvement,84497,15087,17.86
9,credit_card,285708,48650,17.03


In [28]:
query = """
SELECT 
    purpose,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate
FROM loans
GROUP BY purpose
HAVING COUNT(*) >= 1000
ORDER BY default_rate DESC;
"""

purpose_analysis = pd.read_sql_query(query, conn)

print(purpose_analysis.to_string(index=False))

           purpose  total_loans  defaulted_loans  default_rate
    small_business        15010             4465         29.75
            moving         9173             2151         23.45
           medical        15024             3292         21.91
             house         6967             1513         21.72
debt_consolidation       757610           161058         21.26
             other        74937            15867         21.17
          vacation         8732             1680         19.24
    major_purchase        28328             5304         18.72
  home_improvement        84497            15087         17.86
       credit_card       285708            48650         17.03
               car        14121             2068         14.64
           wedding         2294              279         12.16


In [29]:
purpose_analysis[
    ["purpose", "total_loans", "defaulted_loans", "default_rate"]
].to_string(index=False)

'           purpose  total_loans  defaulted_loans  default_rate\n    small_business        15010             4465         29.75\n            moving         9173             2151         23.45\n           medical        15024             3292         21.91\n             house         6967             1513         21.72\ndebt_consolidation       757610           161058         21.26\n             other        74937            15867         21.17\n          vacation         8732             1680         19.24\n    major_purchase        28328             5304         18.72\n  home_improvement        84497            15087         17.86\n       credit_card       285708            48650         17.03\n               car        14121             2068         14.64\n           wedding         2294              279         12.16'

In [30]:
purpose_analysis = pd.read_sql_query(query, conn)
print(purpose_analysis.to_string(index=False))

           purpose  total_loans  defaulted_loans  default_rate
    small_business        15010             4465         29.75
            moving         9173             2151         23.45
           medical        15024             3292         21.91
             house         6967             1513         21.72
debt_consolidation       757610           161058         21.26
             other        74937            15867         21.17
          vacation         8732             1680         19.24
    major_purchase        28328             5304         18.72
  home_improvement        84497            15087         17.86
       credit_card       285708            48650         17.03
               car        14121             2068         14.64
           wedding         2294              279         12.16


Query 5 — Default Rate by Home Ownership

In [31]:
query = """
SELECT 
    home_ownership,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate
FROM loans
GROUP BY home_ownership
HAVING COUNT(*) >= 1000
ORDER BY default_rate DESC;
"""

home_analysis = pd.read_sql_query(query, conn)

print(home_analysis.to_string(index=False))

home_ownership  total_loans  defaulted_loans  default_rate
          RENT       517821           120909         23.35
           OWN       139849            29016         20.75
      MORTGAGE       645509           111675         17.30


### Business Insight — Default Rate by Home Ownership

Default rates differ across home-ownership categories.

- RENT borrowers have the highest observed default rate at 23.35%.
- OWN borrowers have a default rate of 20.75%.
- MORTGAGE borrowers have the lowest observed default rate at 17.30%.

Home-ownership status can therefore be considered as one of the variables for borrower risk segmentation.

Query 6: Verification Status

In [32]:
query = """
SELECT 
    verification_status,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate
FROM loans
GROUP BY verification_status
ORDER BY default_rate DESC;
"""

verification_analysis = pd.read_sql_query(query, conn)

print(verification_analysis.to_string(index=False))

verification_status  total_loans  defaulted_loans  default_rate
           Verified       407689            97593         23.94
    Source Verified       503737           106388         21.12
       Not Verified       392212            57705         14.71


Query 7 — Average Loan Amount by Grade

In [33]:
query = """
SELECT 
    grade,
    COUNT(*) AS total_loans,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate
FROM loans
GROUP BY grade
ORDER BY grade;
"""

grade_financial_analysis = pd.read_sql_query(query, conn)
grade_financial_analysis

,grade,total_loans,avg_loan_amount,avg_interest_rate
0,A,226245,13874.88,7.12
1,B,380158,13227.96,10.69
2,C,369937,14177.07,14.02
3,D,195288,15262.72,17.70
4,E,91574,17639.89,21.10
5,F,31485,19102.16,24.90
6,G,8951,20608.21,27.69


Query 8 — Interest Rate vs Default

In [34]:
query = """
SELECT
    CASE
        WHEN int_rate < 10 THEN '<10%'
        WHEN int_rate < 15 THEN '10-15%'
        WHEN int_rate < 20 THEN '15-20%'
        WHEN int_rate < 25 THEN '20-25%'
        ELSE '25%+'
    END AS interest_rate_band,
    
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(
        100.0 * SUM("default") / COUNT(*), 2
    ) AS default_rate

FROM loans

GROUP BY interest_rate_band

ORDER BY default_rate DESC;
"""

interest_analysis = pd.read_sql_query(query, conn)
interest_analysis

,interest_rate_band,total_loans,defaulted_loans,default_rate
0,25%+,25327,12123,47.87
1,20-25%,81478,31365,38.50
2,15-20%,297389,88078,29.62
3,10-15%,538051,100383,18.66
4,<10%,361393,29737,8.23


Query 9 — DTI vs Default

In [35]:
query = """
SELECT
    CASE
        WHEN dti < 10 THEN '<10'
        WHEN dti < 20 THEN '10-20'
        WHEN dti < 30 THEN '20-30'
        WHEN dti < 40 THEN '30-40'
        ELSE '40+'
    END AS dti_band,

    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(
        100.0 * SUM("default") / COUNT(*), 2
    ) AS default_rate

FROM loans

GROUP BY dti_band

ORDER BY default_rate DESC;
"""

dti_analysis = pd.read_sql_query(query, conn)
dti_analysis

,dti_band,total_loans,defaulted_loans,default_rate
0,40+,6414,2001,31.20
1,30-40,117751,34610,29.39
2,20-30,396273,91888,23.19
3,10-20,545004,97743,17.93
4,<10,238196,35444,14.88


Query 10 — Employment Length vs Default

In [36]:
query = """
SELECT
    emp_length,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(
        100.0 * SUM("default") / COUNT(*), 2
    ) AS default_rate

FROM loans

GROUP BY emp_length

ORDER BY default_rate DESC;
"""

employment_analysis = pd.read_sql_query(query, conn)
employment_analysis

,emp_length,total_loans,defaulted_loans,default_rate
0,None,75457,20397,27.03
1,< 1 year,104552,21622,20.68
2,1 year,85678,17692,20.65
3,3 years,104204,20937,20.09
4,8 years,59127,11864,20.07
5,9 years,49504,9912,20.02
6,2 years,117825,23482,19.93
7,4 years,78033,15526,19.90
8,5 years,81623,16077,19.70
9,7 years,58148,11385,19.58


Query 11 — State-wise Default Risk

In [37]:
query = """
SELECT
    addr_state,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(
        100.0 * SUM("default") / COUNT(*), 2
    ) AS default_rate

FROM loans

GROUP BY addr_state

HAVING COUNT(*) >= 1000

ORDER BY default_rate DESC;
"""

state_analysis = pd.read_sql_query(query, conn)
state_analysis

,addr_state,total_loans,defaulted_loans,default_rate
0,MS,6316,1655,26.20
1,NE,3422,870,25.42
2,AR,9708,2346,24.17
3,AL,16130,3829,23.74
4,OK,11846,2809,23.71
5,LA,15021,3516,23.41
6,NY,106387,23562,22.15
7,NV,19652,4334,22.05
8,IN,21020,4543,21.61
9,TN,19667,4248,21.60


Query 12 — High-Risk Loan Segment

In [38]:
query = """
SELECT
    grade,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(
        100.0 * SUM("default") / COUNT(*), 2
    ) AS default_rate,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount

FROM loans

GROUP BY grade

HAVING COUNT(*) >= 1000

ORDER BY default_rate DESC;
"""

risk_segment_analysis = pd.read_sql_query(query, conn)
risk_segment_analysis

,grade,total_loans,defaulted_loans,default_rate,avg_interest_rate,avg_loan_amount
0,G,8951,4482,50.07,27.69,20608.21
1,F,31485,14265,45.31,24.90,19102.16
2,E,91574,35368,38.62,21.10,17639.89
3,D,195288,59449,30.44,17.70,15262.72
4,C,369937,83271,22.51,14.02,14177.07
5,B,380158,51083,13.44,10.69,13227.96
6,A,226245,13768,6.09,7.12,13874.88


Query 13 — Top High-Risk Combination

In [39]:
query = """
SELECT
    grade,
    home_ownership,
    COUNT(*) AS total_loans,
    SUM("default") AS defaulted_loans,
    ROUND(
        100.0 * SUM("default") / COUNT(*), 2
    ) AS default_rate

FROM loans

GROUP BY grade, home_ownership

HAVING COUNT(*) >= 1000

ORDER BY default_rate DESC

LIMIT 15;
"""

top_risk_segments = pd.read_sql_query(query, conn)
top_risk_segments

,grade,home_ownership,total_loans,defaulted_loans,default_rate
0,G,RENT,3803,2096,55.11
1,G,OWN,1061,534,50.33
2,F,RENT,13688,6708,49.01
3,G,MORTGAGE,4082,1850,45.32
4,F,OWN,3451,1555,45.06
5,E,RENT,38961,16586,42.57
6,F,MORTGAGE,14332,5997,41.84
7,E,OWN,9985,3823,38.29
8,E,MORTGAGE,42601,14948,35.09
9,D,RENT,84927,28417,33.46


Query 14 — Overall Business Summary

In [40]:
query = """
SELECT
    COUNT(*) AS total_loans,
    SUM("default") AS total_defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS overall_default_rate,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(AVG(annual_inc), 2) AS avg_annual_income,
    ROUND(AVG(dti), 2) AS avg_dti
FROM loans;
"""

business_summary = pd.read_sql_query(query, conn)
business_summary

,total_loans,total_defaults,overall_default_rate,avg_loan_amount,avg_interest_rate,avg_annual_income,avg_dti
0,1303638,261686,20.07,14416.84,13.26,76158.75,18.26


# SQL Analytics — Key Business Insights

## Overall Portfolio
- Total loans and overall default rate were calculated.
- Default rate is approximately 20% across the analyzed portfolio.

## Credit Grade
- Default risk increases substantially from Grade A toward Grade G.
- Grade is an important risk segmentation variable.

## Home Ownership
- RENT borrowers showed a higher observed default rate than OWN and MORTGAGE borrowers.
- Home ownership can be useful for borrower segmentation.

## Loan Purpose
- Default rates vary across different loan purposes.
- Purpose can provide additional context for risk assessment.

## Interest Rate
- Default risk varies across interest-rate bands.
- Higher interest-rate segments should be examined carefully during credit-risk assessment.

## Debt-to-Income Ratio
- DTI bands can be compared to identify higher-risk borrower segments.

## Risk Segmentation
- Combining grade and home ownership helps identify specific borrower segments with relatively high observed default rates.

## Business Value
SQL analytics provides interpretable portfolio-level insights that can support:
- Risk segmentation
- Credit policy decisions
- Portfolio monitoring
- High-risk borrower identification
- Dashboard reporting

In [41]:
conn.close()